# yolov8_cpp — real-GPU check (Colab)

Verifies the `bk::` device seam (`pure/backend.hpp`) on an **actual CUDA GPU**.
The same source runs on CPU locally; here it is built with `nvcc -DUSE_CUDA`.

**First: Runtime → Change runtime type → Hardware accelerator = GPU (T4 is fine).**


### 1. Confirm the GPU + nvcc


In [ ]:
!nvidia-smi -L
!nvcc --version


### 2. Clone the repo


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp


### 3. Build the seam demo for the GPU and run it
`m17_gpu.cpp` exercises `bk::Buffer`, `bk::parallel_for`, and `bk::gemm` — the exact
primitives conv/matmul route through. On the GPU path it should print `backend: CUDA (GPU)`.


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 pure/m17_gpu.cpp -o m17_gpu
!./m17_gpu


### 4. Compare with the CPU build (same source)
Proves the *one* source compiles and gives identical results on both backends.


In [ ]:
!g++ -std=c++20 -O2 -fopenmp pure/m17_gpu.cpp -o m17_cpu
!./m17_cpu


### 5. (stretch) Build the full engine with nvcc
Compiles a full-forward test as CUDA (conv im2col-GEMM runs on the GPU via the seam).
This is the bigger step — nvcc must accept the C++20 host code. If it errors, paste the
output back to Claude and it will fix the seam/build.

First regenerate the weights + reference (needs Ultralytics):


In [ ]:
!pip -q install ultralytics==8.4.104 >/dev/null 2>&1
!python pure/ref/export_net.py 64 2>/dev/null | tail -1


In [ ]:
# full yolov8n forward built as CUDA; compares to the PyTorch reference
!nvcc -x cu -std=c++20 --extended-lambda -DUSE_CUDA -O2 pure/m3c_forward.cpp -o m3c_gpu 2>&1 | tail -20
!./m3c_gpu 2>&1 | tail -5


### What success looks like
- Cell 3: `backend: CUDA (GPU)` and `M17 OK` → the seam runs on real CUDA.
- Cell 4: `backend: CPU ...` and `M17 OK` → same source, same result on CPU.
- Cell 6 (stretch): `PURE ENGINE == yolov8n forward` → the whole net runs on the GPU.

The same applies to `yolov5_cpp` and `yolov11_cpp` (identical `pure/backend.hpp`).
